### 다중분류_컬러이미지
- CIFAR 10 dataset
  - train data : 50,000
  - test data : 10,000
  - shape : 32 x 32 (color image)
  - channel : 3 (RGB)
  - target classes : 10
    - airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# sys.path.append('/content/drive/MyDrive/Colab Notebooks')

from multiclass_functions import *
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import numpy as np
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
BATCH_SIZE = 32
LR = 1e-3
EPOCH = 15
criterion = nn.CrossEntropyLoss()
new_model_train = True

model_type = 'MLP_deep'
dataset = 'CIFAR10'

save_dir = 'results'
os.makedirs(save_dir, exist_ok=True)

save_model_path = f'{save_dir}/{model_type}_{dataset}.pt'

In [4]:
transform = transforms.ToTensor()
train_DS = datasets.CIFAR10(root='data', train=True, transform=transform, download=True)
test_DS = datasets.CIFAR10(root='data', train=False, transform=transform, download=True)

train_DL = DataLoader(train_DS, batch_size= BATCH_SIZE, shuffle=True)
test_DL = DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=True)

100.0%


In [ ]:
print(train_DS)
print(test_DS)
print(test_DS.classes)
print(test_DS.class_to_idx)

In [ ]:
plt.imshow(train_DS.data[99])

In [ ]:
x_batch, y_batch = next(iter(train_DL))
print(x_batch.shape)

plt.imshow(x_batch[0].permute(1,2,0))
'''
• 기존 형태 (0, 1, 2): PyTorch에서 이미지 텐서는 보통 (Channels, Height, Width) 순서로 저장됩니다. 
  예를 들어 3 x 224 x 224 형태입니다.
• 변경 후 (1, 2, 0): 기존 1번 축(Height), 2번 축(Width), 0번 축(Channels) 순서로 차원을 재배치합니다. 
  결과적으로 224 x 224 x 3 형태가 됩니다.
'''
print(test_DS.classes[y_batch[0]])

In [ ]:
print(type(train_DS.data))
print(train_DS.data.shape)
print(train_DS.data.dtype)
print(train_DS.data[0][0][0][0])
print('-'*30)

print(type(x_batch))
print(x_batch.shape)
print(x_batch.dtype)

In [ ]:
class MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Sequential(
        nn.Linear(3*32*32, 30),
        nn.BatchNorm1d(30),  
        # 학습하는 데이터의 안정성을 제고하기 위한 정규화
        # 전체 데이터가 아닌 배치 데이터만으로 정규화
        # 은닉층에서 학습함수와 활성함수 사이에 위치
        nn.ReLU(),
        nn.Linear(30, 10)
    )

  def forward(self, x):
    x = torch.flatten(x, start_dim=1)
    x = self.linear(x)
    return x

# 은닉층 뉴런의 개수를 늘림: 30 -> 100
class MLP_shallow(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Sequential(
        nn.Linear(3*32*32, 100),  
        nn.BatchNorm1d(100),  
        nn.ReLU(),
        nn.Linear(100, 10)
    )

  def forward(self, x):
    x = torch.flatten(x, start_dim=1)
    x = self.linear(x)
    return x

# 은닉층의 층수를 늘림: 4 -> 34
'''
layers = []

for _ in range(10):
    for i in [nn.Linear(75, 75),
              nn.BatchNorm1d(75),
              nn.ReLU()]:

        layers.append(i)
'''
class MLP_deep(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Sequential(
        nn.Linear(3*32*32, 75),
        nn.BatchNorm1d(75),
        nn.ReLU(),
        *[i for _ in range(10) for i in [nn.Linear(75, 75),
                                         nn.BatchNorm1d(75), 
                                         nn.ReLU()]],
        nn.Linear(75, 10)
    )

  def forward(self, x):
    x = torch.flatten(x, start_dim=1)
    x = self.linear(x)
    return x

In [ ]:
model = globals()[model_type]().to(device)
print(model)
# model_type 은 하이퍼 파라미터 셀에서 model_type = 'MLP' 로 정의했기 때문에 변경희망시
# model_type = 'MLP_shallow' 로 변경하여 다시 셀 2 개를 실행하면 됨

x_batch, _ = next(iter(train_DL))
print(x_batch.shape)
print(model(x_batch.to(device)).shape)

In [ ]:
# if new_model_train == True:
if new_model_train:
  optimizer = optim.Adam(model.parameters(), lr=LR)
  loss_history = Train(model, train_DL, criterion, optimizer, EPOCH)

  torch.save(model, save_model_path)

  plt.figure()
  plt.plot(range(1, EPOCH+1), loss_history)
  plt.xlabel('ep')
  plt.ylabel('loss')
  plt.title('TR LOSS')
  plt.grid()

In [ ]:
load_model = torch.load(save_model_path, map_location= device, weights_only=False)

In [ ]:
Test(load_model, test_DL)
print(count_params(load_model))

In [ ]:
Test_plot(load_model, test_DL)

In [ ]:
confusion = get_conf(load_model, test_DL)

plot_confusion_matrix(confusion)